# Advanced `__new__` — Tutorial-Style Problems with Solutions

This notebook is a second, independent advanced practice notebook on Python's `__new__`.

The style is intentionally tutorial-like:

- concepts are introduced in small steps
- examples are built incrementally
- problems are broken into logical sub-problems
- many markdown cells explain *why* something happens
- predictions come before execution
- complete solutions follow each challenge
- later sections combine several ideas at once

The goal is not just to memorize `__new__`, but to learn how to *reason about object creation*.

## 1. First, recall the object creation pipeline

When we write:

```python
obj = SomeClass(arg1, arg2)
```

there are two important phases:

1. Python asks `__new__` to create or return an object.
2. If that returned object is an instance of `SomeClass`, Python normally calls `__init__` on it.

A useful mental model is:

```python
obj = SomeClass.__new__(SomeClass, arg1, arg2)

if isinstance(obj, SomeClass):
    SomeClass.__init__(obj, arg1, arg2)
```

This is not the whole implementation detail of Python's metaclass machinery, but it is a very useful model for predicting behavior.

### A very small tracing example

Before looking at difficult cases, let us make the two phases visible.

In [56]:
class Demo:
    def __new__(cls, value):
        print("1. __new__ is running")
        obj = super().__new__(cls)
        print("2. __new__ is returning", obj)
        return obj

    def __init__(self, value):
        print("3. __init__ is running")
        self.value = value
        print("4. value stored")

d = Demo(10)
print("5. final object:", d.__dict__)

1. __new__ is running
2. __new__ is returning <__main__.Demo object at 0x000001D55FB7F8C0>
3. __init__ is running
4. value stored
5. final object: {'value': 10}


Notice that `__new__` receives `cls`, not `self`.

That makes sense: there is no instance yet.

The purpose of `__new__` is to produce the instance that may later become `self` inside `__init__`.

# Problem 1 — What happens if `__new__` returns `None`?

We will start with a deceptively small problem.

Consider this class:

```python
class Broken:
    def __new__(cls):
        print("creating...")
        super().__new__(cls)

    def __init__(self):
        print("initializing...")
```

Before running anything, answer:

1. What does `Broken()` return?
2. Does `__init__` run?
3. Why?

## Step 1 — Remember Python's default return value

A Python function without an explicit `return` statement returns `None`.

So although `super().__new__(cls)` creates an object, we immediately lose it because we never return it.

In [57]:
class Broken:
    def __new__(cls):
        print("creating...")
        super().__new__(cls)

    def __init__(self):
        print("initializing...")

result = Broken()

print("result:", result)
print("type:", type(result))

creating...
result: None
type: <class 'NoneType'>


## Solution

`Broken()` returns `None`.

`__init__` does **not** run because the object returned by `__new__` is `None`, which is not an instance of `Broken`.

The correct implementation is:

In [58]:
class Fixed:
    def __new__(cls):
        print("creating...")
        obj = super().__new__(cls)
        return obj

    def __init__(self):
        print("initializing...")

result = Fixed()
assert isinstance(result, Fixed)

creating...
initializing...


### Best-practice takeaway

When overriding `__new__`, always be very deliberate about the value you return.

The return value is not just a detail — it determines the actual result of the class call.

# Problem 2 — Returning an object of another type

Now let us deliberately return the "wrong" type.

We want a class call like:

```python
MaybeInteger("123")
```

to return the integer `123`.

But:

```python
MaybeInteger("hello")
```

should return an actual `MaybeInteger` instance.

## Step 1 — Think about the two branches

If the text contains only digits, `__new__` can simply return an `int`.

Otherwise it should ask the normal inheritance chain to create a `MaybeInteger`.

In [59]:
class MaybeInteger:
    def __new__(cls, text):
        if str(text).isdigit():
            return int(text)
        return super().__new__(cls)

    def __init__(self, text):
        print("__init__ called for", text)
        self.text = str(text)

## Step 2 — Predict the output

Which construction below causes `__init__` to run?

```python
a = MaybeInteger("123")
b = MaybeInteger("hello")
```

In [60]:
a = MaybeInteger("123")
b = MaybeInteger("hello")

print("a:", a, type(a))
print("b:", b, type(b))
print("b.__dict__:", b.__dict__)

__init__ called for hello
a: 123 <class 'int'>
b: <__main__.MaybeInteger object at 0x000001D55FB7FCB0> <class '__main__.MaybeInteger'>
b.__dict__: {'text': 'hello'}


## Solution

`a` is an `int`, so `MaybeInteger.__init__` is skipped.

`b` is a `MaybeInteger`, so Python continues with `MaybeInteger.__init__`.

This behavior is one of the most important things to understand about `__new__`.

# Problem 3 — Why immutable subclasses need `__new__`

Suppose we want this:

```python
SquaredInt(5) == 25
```

and we also want the result to be an instance of `SquaredInt`.

A first instinct might be to try `__init__`.

In [61]:
class WrongSquaredInt(int):
    def __init__(self, value):
        print("Trying to change", self, "into", value ** 2)

## Step 1 — Why can't `__init__` replace the integer value?

By the time `__init__` runs, the integer object already exists.

Integers are immutable.

We need to create the integer with the desired value *during construction*.

In [62]:
class SquaredInt(int):
    def __new__(cls, value):
        return super().__new__(cls, value ** 2)

x = SquaredInt(5)

print(x)
print(type(x))
print(isinstance(x, int))
print(isinstance(x, SquaredInt))

assert x == 25

25
<class '__main__.SquaredInt'>
True
True


## Solution

For immutable built-in subclasses, `__new__` is where the underlying immutable value is usually selected.

That same principle applies to types such as:

- `int`
- `float`
- `str`
- `tuple`

# Problem 4 — Build a normalized string type

We will now build an immutable `Slug` string.

Requirements:

```python
Slug("  Hello   WORLD  ")
```

should become:

```text
hello-world
```

Rules:

1. trim outer whitespace
2. lowercase the text
3. split on arbitrary whitespace
4. join words with `-`
5. reject an empty result

## Step 1 — Write the normalization logic before subclassing

It is often useful to solve the transformation independently first.

In [63]:
def normalize_slug(value):
    words = str(value).strip().lower().split()
    return "-".join(words)

print(normalize_slug("  Hello   WORLD  "))

hello-world


## Step 2 — Move that transformation into `__new__`

Because `str` is immutable, the normalized string must be passed to `str.__new__`.

In [64]:
class Slug(str):
    def __new__(cls, value):
        normalized = "-".join(str(value).strip().lower().split())

        if not normalized:
            raise ValueError("slug cannot be empty")

        return super().__new__(cls, normalized)

## Step 3 — Test both the value and the type

A common mistake is to test only that the value looks correct.

For `__new__` code, the concrete type matters too.

In [65]:
s = Slug("  Hello   WORLD  ")

print("value:", s)
print("type:", type(s))

assert s == "hello-world"
assert type(s) is Slug
assert isinstance(s, str)

value: hello-world
type: <class '__main__.Slug'>


# Problem 5 — Add subclass behavior without breaking construction

Now create:

```python
class ArticleSlug(Slug):
    pass
```

Question:

Will `ArticleSlug("Hello World")` produce a `Slug` or an `ArticleSlug`?

In [66]:
class ArticleSlug(Slug):
    pass

a = ArticleSlug("Hello World")

print(a)
print(type(a))

assert type(a) is ArticleSlug

hello-world
<class '__main__.ArticleSlug'>


## Why?

Inside `Slug.__new__`, the `cls` argument is the concrete class being constructed.

When we call:

```python
super().__new__(cls, normalized)
```

we preserve the requested subclass.

Hard-coding `Slug` there would be much less inheritance-friendly.

# Problem 6 — A tuple subclass built step by step

We want an immutable 2D point:

```python
Point2D(3, 4)
```

that behaves like the tuple:

```python
(3.0, 4.0)
```

but also exposes `.x` and `.y`.

## Step 1 — Remember how a tuple subclass is created

`tuple.__new__` expects an iterable containing the tuple's elements.

So the constructor should eventually do something like:

```python
super().__new__(cls, (x, y))
```

In [67]:
class Point2D(tuple):
    def __new__(cls, x, y):
        values = (float(x), float(y))
        return super().__new__(cls, values)

    @property
    def x(self):
        return self[0]

    @property
    def y(self):
        return self[1]

## Step 2 — Check tuple behavior

We want both worlds:

- custom semantic type
- normal tuple operations

In [68]:
p = Point2D(3, 4)

print("p:", p)
print("x:", p.x)
print("y:", p.y)
print("len:", len(p))
print("first:", p[0])

assert p == (3.0, 4.0)
assert isinstance(p, tuple)
assert isinstance(p, Point2D)

p: (3.0, 4.0)
x: 3.0
y: 4.0
len: 2
first: 3.0


# Problem 7 — Add validation before immutable construction

Now strengthen `Point2D`.

Coordinates must be finite.

Reject:

- `inf`
- `-inf`
- `nan`

## Step 1 — Why validate before calling the parent `__new__`?

Because immutable construction should happen only after we know the final stored values are valid.

In [69]:
import math

class FinitePoint2D(tuple):
    def __new__(cls, x, y):
        values = (float(x), float(y))

        if not all(math.isfinite(v) for v in values):
            raise ValueError("coordinates must be finite")

        return super().__new__(cls, values)

    x = property(lambda self: self[0])
    y = property(lambda self: self[1])

In [70]:
good = FinitePoint2D(1, 2)
print(good)

for bad in [
    (float("inf"), 1),
    (1, float("-inf")),
    (float("nan"), 2),
]:
    try:
        FinitePoint2D(*bad)
    except ValueError as ex:
        print("rejected:", bad, "->", ex)

(1.0, 2.0)
rejected: (inf, 1) -> coordinates must be finite
rejected: (1, -inf) -> coordinates must be finite
rejected: (nan, 2) -> coordinates must be finite


# Problem 8 — Cooperative inheritance

We now move to a more subtle topic.

Suppose both a parent and a child want to participate in object creation.

A fragile implementation is:

```python
class Parent:
    def __new__(cls, value):
        return object.__new__(cls)

class Child(Parent):
    def __new__(cls, value):
        return object.__new__(cls)
```

The child directly skips `Parent.__new__`.

## Step 1 — Make the call chain visible

In [71]:
class Parent:
    def __new__(cls, value):
        print("Parent.__new__")
        return super().__new__(cls)

class Child(Parent):
    def __new__(cls, value):
        print("Child.__new__")
        return super().__new__(cls, value)

c = Child(10)

Child.__new__
Parent.__new__


## Solution

The output shows:

```text
Child.__new__
Parent.__new__
```

The child delegates to the *next implementation in the MRO*.

This is the same cooperative inheritance principle we use with `super().__init__`.

# Problem 9 — Preserve the most-derived class

We will extend the inheritance example.

Suppose a base class adds an attribute during `__new__`.

Will a subclass still be created correctly?

In [72]:
class BaseEntity:
    def __new__(cls, name):
        obj = super().__new__(cls)
        obj.created_as = cls.__name__
        return obj

    def __init__(self, name):
        self.name = name

class UserEntity(BaseEntity):
    pass

u = UserEntity("Ada")

print(type(u))
print(u.created_as)
print(u.name)

assert type(u) is UserEntity
assert u.created_as == "UserEntity"

<class '__main__.UserEntity'>
UserEntity
Ada


## Key observation

The base class should normally create:

```python
super().__new__(cls)
```

not:

```python
super().__new__(BaseEntity)
```

The first form respects subclasses.

# Problem 10 — A simple singleton

We now use `__new__` for identity control.

Goal:

```python
a = Singleton()
b = Singleton()

a is b
```

should be `True`.

In [73]:
class Singleton:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

a = Singleton()
b = Singleton()

print(a is b)
assert a is b

True


At first sight, this looks complete.

But there is an important trap.

# Problem 11 — The singleton initialization trap

Add an `__init__` that stores a value.

In [74]:
class BrokenConfig:
    _instance = None

    def __new__(cls, value):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value):
        print("__init__ called with", value)
        self.value = value

a = BrokenConfig("first")
b = BrokenConfig("second")

print("same object?", a is b)
print("a.value:", a.value)
print("b.value:", b.value)

__init__ called with first
__init__ called with second
same object? True
a.value: second
b.value: second


## What happened?

`__new__` returned the same object twice.

But both class calls still returned an instance of `BrokenConfig`, so `__init__` ran twice.

The second call mutated the already-existing singleton.

## Step 2 — Guard initialization

In [75]:
class Config:
    _instance = None

    def __new__(cls, value):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    def __init__(self, value):
        if self._initialized:
            return

        self.value = value
        self._initialized = True

a = Config("first")
b = Config("second")

print(a is b)
print(a.value)

assert a is b
assert a.value == "first"

True
first


### Best-practice note

A singleton is not automatically the best design.

For many applications, clearer alternatives include:

- a module-level object
- explicit dependency injection
- a factory with controlled lifetime

But the pattern is still valuable for understanding `__new__`.

# Problem 12 — Interning / flyweight objects

Singletons reuse one object globally.

Interning is more flexible:

- one object per unique key
- repeated equivalent construction returns the same identity

We want:

```python
Symbol("Alpha") is Symbol(" alpha ")
```

to be `True`.

## Step 1 — Normalize the key

We should define identity based on normalized input.

In [76]:
def normalize_symbol(name):
    return str(name).strip().casefold()

print(normalize_symbol(" Alpha "))

alpha


## Step 2 — Create a cache

In [77]:
class Symbol:
    _cache = {}

    def __new__(cls, name):
        key = str(name).strip().casefold()

        if key in cls._cache:
            return cls._cache[key]

        obj = super().__new__(cls)
        obj.name = key
        cls._cache[key] = obj
        return obj

## Step 3 — Test identity

In [78]:
a = Symbol("Alpha")
b = Symbol(" alpha ")
c = Symbol("Beta")

print(a is b)
print(a is c)
print(a.name, c.name)

assert a is b
assert a is not c

True
False
alpha beta


# Problem 13 — Avoid a global-memory cache

The previous cache keeps strong references forever.

For long-running programs, that can be undesirable.

Python provides `weakref.WeakValueDictionary`, which allows cached objects to disappear when nothing else references them.

In [79]:
import weakref

class WeakSymbol:
    _cache = weakref.WeakValueDictionary()

    def __new__(cls, name):
        key = str(name).strip().casefold()

        existing = cls._cache.get(key)
        if existing is not None:
            return existing

        obj = super().__new__(cls)
        obj.name = key
        cls._cache[key] = obj
        return obj

x = WeakSymbol("alpha")
y = WeakSymbol("ALPHA")

print(x is y)
assert x is y

True


# Problem 14 — Caches and subclasses

Suppose we subclass `WeakSymbol`.

Should:

```python
WeakSymbol("x")
SpecialSymbol("x")
```

return the same object?

Usually, no.

The concrete type should be part of the cache key.

In [80]:
import weakref

class TypedSymbol:
    _cache = weakref.WeakValueDictionary()

    def __new__(cls, name):
        normalized = str(name).strip().casefold()
        key = (cls, normalized)

        existing = cls._cache.get(key)
        if existing is not None:
            return existing

        obj = super().__new__(cls)
        obj.name = normalized
        cls._cache[key] = obj
        return obj

class SpecialSymbol(TypedSymbol):
    pass

a = TypedSymbol("x")
b = SpecialSymbol("x")
c = SpecialSymbol(" X ")

print(a is b)
print(b is c)

assert a is not b
assert b is c

False
True


# Problem 15 — A validated integer domain type

Let us build `Port(int)`.

Requirements:

- value must be integer-convertible
- valid range is `1..65535`
- booleans should be rejected explicitly
- result remains an `int`

## Step 1 — Why reject booleans?

In Python:

```python
isinstance(True, int)
```

is `True`.

For many domain models, `True` should not silently become port `1`.

In [81]:
class Port(int):
    def __new__(cls, value):
        if isinstance(value, bool):
            raise TypeError("bool is not a valid port")

        value = int(value)

        if not 1 <= value <= 65535:
            raise ValueError("port must be in range 1..65535")

        return super().__new__(cls, value)

In [82]:
for value in [80, "443", 65535]:
    p = Port(value)
    print(p, type(p))

for value in [0, 65536, True]:
    try:
        Port(value)
    except (TypeError, ValueError) as ex:
        print("rejected:", value, "->", ex)

80 <class '__main__.Port'>
443 <class '__main__.Port'>
65535 <class '__main__.Port'>
rejected: 0 -> port must be in range 1..65535
rejected: 65536 -> port must be in range 1..65535
rejected: True -> bool is not a valid port


# Problem 16 — Add metadata to an immutable subclass

We can sometimes attach normal attributes to immutable subclasses.

Create `TaggedInt(value, tag)`.

In [83]:
class TaggedInt(int):
    def __new__(cls, value, tag):
        obj = super().__new__(cls, int(value))
        obj.tag = str(tag)
        return obj

x = TaggedInt(7, "lucky")

print(x)
print(x.tag)
print(x.__dict__)

7
lucky
{'tag': 'lucky'}


The integer value is immutable, but the subclass instance can still carry additional attributes if the type layout allows it.

# Problem 17 — Pickling a custom immutable constructor

Now consider saving and restoring `TaggedInt`.

Pickle needs enough information to reconstruct the object.

A convenient hook is `__getnewargs_ex__`.

In [84]:
import pickle

class PickleTaggedInt(int):
    def __new__(cls, value, tag):
        obj = super().__new__(cls, int(value))
        obj.tag = str(tag)
        return obj

    def __getnewargs_ex__(self):
        return ((int(self), self.tag), {})

original = PickleTaggedInt(42, "answer")
blob = pickle.dumps(original)
restored = pickle.loads(blob)

print(restored, restored.tag, type(restored))

assert restored == 42
assert restored.tag == "answer"
assert type(restored) is PickleTaggedInt

42 answer <class '__main__.PickleTaggedInt'>


# Problem 18 — `__slots__` and `__new__`

Now let us combine controlled construction with slotted storage.

We want a class with:

- no `__dict__`
- an original string
- a normalized string

In [85]:
class NormalizedName:
    __slots__ = ("original", "normalized")

    def __new__(cls, value):
        obj = super().__new__(cls)
        obj.normalized = str(value).strip().casefold()
        return obj

    def __init__(self, value):
        self.original = str(value)

n = NormalizedName("  PyThOn  ")

print(n.original)
print(n.normalized)
print("has __dict__?", hasattr(n, "__dict__"))

assert not hasattr(n, "__dict__")

  PyThOn  
python
has __dict__? False


# Problem 19 — Weak references plus `__slots__`

A slotted class does not automatically support weak references.

To use it inside a `WeakValueDictionary`, add `__weakref__` to `__slots__`.

In [86]:
import weakref

class SlottedToken:
    __slots__ = ("text", "__weakref__")
    _cache = weakref.WeakValueDictionary()

    def __new__(cls, text):
        key = str(text).strip()

        existing = cls._cache.get(key)
        if existing is not None:
            return existing

        obj = super().__new__(cls)
        obj.text = key
        cls._cache[key] = obj
        return obj

t1 = SlottedToken("abc")
t2 = SlottedToken("abc")

print(t1 is t2)
assert t1 is t2

True


# Problem 20 — Design an immutable `Money` object

We now build a more realistic value object.

Represent money as:

```python
Money("usd", 125)
```

meaning 125 cents of USD.

The object should be immutable and tuple-like.

## Step 1 — Decide the canonical representation

We will store exactly:

```python
("USD", 125)
```

inside the tuple.

In [87]:
class Money(tuple):
    def __new__(cls, currency, cents):
        currency = str(currency).strip().upper()

        if len(currency) != 3 or not currency.isalpha():
            raise ValueError("currency must be a 3-letter alphabetic code")

        if isinstance(cents, bool):
            raise TypeError("cents cannot be bool")

        cents = int(cents)

        return super().__new__(cls, (currency, cents))

    @property
    def currency(self):
        return self[0]

    @property
    def cents(self):
        return self[1]

## Step 2 — Verify canonicalization

In [88]:
m = Money(" usd ", "125")

print(m)
print(m.currency)
print(m.cents)
print(type(m))

assert m == ("USD", 125)

('USD', 125)
USD
125
<class '__main__.Money'>


## Step 3 — Add arithmetic carefully

Adding money only makes sense when the currencies match.

When constructing the result, use `type(self)` rather than hard-coding `Money`.

That lets subclasses remain subclasses.

In [89]:
class Money(tuple):
    def __new__(cls, currency, cents):
        currency = str(currency).strip().upper()

        if len(currency) != 3 or not currency.isalpha():
            raise ValueError("currency must be a 3-letter alphabetic code")

        if isinstance(cents, bool):
            raise TypeError("cents cannot be bool")

        return super().__new__(cls, (currency, int(cents)))

    currency = property(lambda self: self[0])
    cents = property(lambda self: self[1])

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented

        if self.currency != other.currency:
            raise ValueError("cannot add different currencies")

        return type(self)(self.currency, self.cents + other.cents)

In [90]:
m1 = Money("USD", 100)
m2 = Money("usd", 250)
m3 = m1 + m2

print(m3)
assert m3 == ("USD", 350)

try:
    Money("USD", 100) + Money("EUR", 100)
except ValueError as ex:
    print("expected:", ex)

('USD', 350)
expected: cannot add different currencies


# Problem 21 — Preserve subclass type in operations

Create:

```python
class RewardMoney(Money):
    pass
```

Then add two `RewardMoney` instances.

What type should the result have?

In [91]:
class RewardMoney(Money):
    pass

a = RewardMoney("EUR", 50)
b = RewardMoney("EUR", 75)
c = a + b

print(c)
print(type(c))

assert type(c) is RewardMoney
assert c.cents == 125

('EUR', 125)
<class '__main__.RewardMoney'>


# Problem 22 — Constructor factory behavior

`__new__` can return an object of a completely different class.

We can use that to select a concrete implementation.

For learning purposes, let us build:

```python
Shape("circle", radius=2)
Shape("rectangle", width=3, height=4)
```

In [92]:
import math

class Circle:
    def __init__(self, radius):
        self.radius = float(radius)

    @property
    def area(self):
        return math.pi * self.radius ** 2

class Rectangle:
    def __init__(self, width, height):
        self.width = float(width)
        self.height = float(height)

    @property
    def area(self):
        return self.width * self.height

class Shape:
    def __new__(cls, kind, **kwargs):
        kind = str(kind).strip().casefold()

        if kind == "circle":
            return Circle(**kwargs)

        if kind == "rectangle":
            return Rectangle(**kwargs)

        raise ValueError(f"unknown shape kind: {kind}")

In [93]:
circle = Shape("circle", radius=2)
rectangle = Shape("rectangle", width=3, height=4)

print(type(circle), circle.area)
print(type(rectangle), rectangle.area)

assert type(circle) is Circle
assert type(rectangle) is Rectangle

<class '__main__.Circle'> 12.566370614359172
<class '__main__.Rectangle'> 12.0


## Important design discussion

This works, but it can surprise readers because calling `Shape(...)` does not actually produce a `Shape`.

A named factory is often clearer:

In [94]:
class ShapeFactory:
    @classmethod
    def create(cls, kind, **kwargs):
        kind = str(kind).strip().casefold()

        if kind == "circle":
            return Circle(**kwargs)

        if kind == "rectangle":
            return Rectangle(**kwargs)

        raise ValueError(kind)

shape = ShapeFactory.create("circle", radius=3)
print(type(shape))

<class '__main__.Circle'>


# Problem 23 — Multiple inheritance and cooperative `__new__`

Multiple inheritance is where cooperative `super()` becomes especially important.

We will create two mixins that both want to participate in creation.

In [95]:
class LoggingMixin:
    def __new__(cls, *args, **kwargs):
        print("LoggingMixin.__new__")
        return super().__new__(cls)

class AuditMixin:
    def __new__(cls, *args, **kwargs):
        print("AuditMixin.__new__")
        obj = super().__new__(cls)
        obj.audited = True
        return obj

class Account(LoggingMixin, AuditMixin):
    def __init__(self, name):
        self.name = name

print(Account.__mro__)

a = Account("main")

print(a.name)
print(a.audited)

(<class '__main__.Account'>, <class '__main__.LoggingMixin'>, <class '__main__.AuditMixin'>, <class 'object'>)
LoggingMixin.__new__
AuditMixin.__new__
main
True


## Why did both mixins run?

Each mixin asked `super()` for the next constructor in the MRO.

Neither mixin hard-coded the other.

That makes the implementation composable.

# Problem 24 — Class-specific creation counters

We want every concrete class to count how many instances it has created.

But class attributes are inherited, so we must distinguish:

```python
getattr(cls, "created", 0)
```

from:

```python
cls.__dict__.get("created", 0)
```

In [96]:
class Counted:
    def __new__(cls):
        obj = super().__new__(cls)
        current = cls.__dict__.get("created", 0)
        cls.created = current + 1
        return obj

class A(Counted):
    pass

class B(Counted):
    pass

Counted()
A()
A()
B()

print("Counted:", Counted.created)
print("A:", A.created)
print("B:", B.created)

assert Counted.created == 1
assert A.created == 2
assert B.created == 1

Counted: 1
A: 2
B: 1


# Problem 25 — A canonical semantic version type

Build:

```python
Version("1.4.2")
Version(1, 4, 2)
```

as the same tuple-like representation.

## Step 1 — Accept two constructor forms

In [97]:
class Version(tuple):
    def __new__(cls, *args):
        if len(args) == 1 and isinstance(args[0], str):
            parts = args[0].strip().split(".")
        else:
            parts = args

        if len(parts) != 3:
            raise ValueError("version requires exactly 3 components")

        try:
            values = tuple(int(part) for part in parts)
        except (TypeError, ValueError):
            raise ValueError("version components must be integers") from None

        if any(value < 0 for value in values):
            raise ValueError("version components must be non-negative")

        return super().__new__(cls, values)

    def __str__(self):
        return ".".join(str(x) for x in self)

## Step 2 — Let tuple comparison do the work

In [98]:
v1 = Version("1.4.2")
v2 = Version(1, 5, 0)

print(v1)
print(v2)
print("v1 < v2:", v1 < v2)

assert v1 == (1, 4, 2)
assert v1 < v2

1.4.2
1.5.0
v1 < v2: True


# Problem 26 — Canonical rational numbers

A rational number is a great `__new__` exercise because many different inputs represent the same mathematical value.

Examples:

```python
Rational(2, 4)   -> (1, 2)
Rational(1, -2)  -> (-1, 2)
Rational(-2, -4) -> (1, 2)
```

## Step 1 — Normalize the sign

We choose a canonical rule:

> the denominator is always positive.

## Step 2 — Reduce by greatest common divisor

Use `math.gcd`.

In [99]:
import math

class Rational(tuple):
    def __new__(cls, numerator, denominator=1):
        n = int(numerator)
        d = int(denominator)

        if d == 0:
            raise ZeroDivisionError("denominator cannot be zero")

        if d < 0:
            n = -n
            d = -d

        divisor = math.gcd(n, d)
        n //= divisor
        d //= divisor

        return super().__new__(cls, (n, d))

    numerator = property(lambda self: self[0])
    denominator = property(lambda self: self[1])

    def __repr__(self):
        return f"{type(self).__name__}({self.numerator}, {self.denominator})"

In [100]:
for r in [
    Rational(2, 4),
    Rational(1, -2),
    Rational(-2, -4),
]:
    print(r)

assert Rational(2, 4) == (1, 2)
assert Rational(1, -2) == (-1, 2)
assert Rational(-2, -4) == (1, 2)

Rational(1, 2)
Rational(-1, 2)
Rational(1, 2)


## Step 3 — Add two rationals

Again, preserve subclasses with `type(self)`.

In [101]:
class Rational(tuple):
    def __new__(cls, numerator, denominator=1):
        n = int(numerator)
        d = int(denominator)

        if d == 0:
            raise ZeroDivisionError("denominator cannot be zero")

        if d < 0:
            n, d = -n, -d

        divisor = math.gcd(n, d)
        return super().__new__(cls, (n // divisor, d // divisor))

    numerator = property(lambda self: self[0])
    denominator = property(lambda self: self[1])

    def __add__(self, other):
        if not isinstance(other, Rational):
            return NotImplemented

        n = self.numerator * other.denominator + other.numerator * self.denominator
        d = self.denominator * other.denominator

        return type(self)(n, d)

    def __repr__(self):
        return f"{type(self).__name__}({self.numerator}, {self.denominator})"

result = Rational(1, 2) + Rational(1, 6)

print(result)
assert result == (2, 3)

Rational(2, 3)


# Problem 27 — Recursive construction bug

Consider this implementation:

```python
class Recursive:
    def __new__(cls, value):
        return cls(value)
```

What happens?

`cls(value)` calls the class again.

That calls `__new__` again.

Which calls `cls(value)` again.

And so on.

## Correct principle

Inside `__new__`, delegate to a base constructor:

```python
super().__new__(cls)
```

or, for immutable built-ins:

```python
super().__new__(cls, transformed_value)
```

Do not recursively invoke the same class constructor unless recursion is very intentionally controlled.

In [102]:
class NonRecursive:
    def __new__(cls, value):
        obj = super().__new__(cls)
        return obj

    def __init__(self, value):
        self.value = value

x = NonRecursive(10)
print(x.value)

10


# Problem 28 — `object.__new__` argument mistake

A useful low-level detail:

```python
object.__new__(SomeClass)
```

is valid.

But arbitrary constructor arguments should not be forwarded to `object.__new__`.

In [103]:
class Plain:
    pass

try:
    object.__new__(Plain, 1, 2)
except TypeError as ex:
    print("expected:", ex)

expected: Plain() takes no arguments


This matters when writing inheritance-friendly constructors.

The arguments consumed by your class hierarchy do not necessarily belong to `object.__new__`.

# Problem 29 — Metaclass view of construction

There is another layer above `__new__`.

Calling a class normally invokes the metaclass's `__call__`.

For ordinary classes, the metaclass is `type`.

`type.__call__` coordinates object creation and initialization.

In [104]:
class TraceMeta(type):
    def __call__(cls, *args, **kwargs):
        print("metaclass: before class construction")
        obj = super().__call__(*args, **kwargs)
        print("metaclass: after class construction")
        return obj

class MetaDemo(metaclass=TraceMeta):
    def __new__(cls, value):
        print("MetaDemo.__new__")
        return super().__new__(cls)

    def __init__(self, value):
        print("MetaDemo.__init__")
        self.value = value

m = MetaDemo(100)

metaclass: before class construction
MetaDemo.__new__
MetaDemo.__init__
metaclass: after class construction


The order is:

1. metaclass `__call__`
2. class `__new__`
3. class `__init__`
4. metaclass `__call__` returns the result

This gives a more complete picture of class instantiation.

# Problem 30 — Final mini-project: immutable RGB color

We will combine several ideas:

- multiple constructor forms
- input validation
- immutable tuple storage
- canonical representation
- properties
- hashing

## Step 1 — Desired API

Both of these should work:

```python
Color(255, 128, 0)
Color("#ff8000")
```

Both should represent exactly the same immutable color.

## Step 2 — Parse the hex form

`#RRGGBB` contains three two-character hexadecimal channels.

In [105]:
def parse_hex_color(text):
    text = str(text).strip()

    if len(text) != 7 or not text.startswith("#"):
        raise ValueError("hex color must look like #RRGGBB")

    try:
        return tuple(int(text[i:i+2], 16) for i in (1, 3, 5))
    except ValueError:
        raise ValueError("invalid hexadecimal color") from None

print(parse_hex_color("#ff8000"))

(255, 128, 0)


## Step 3 — Build the tuple subclass

In [106]:
class Color(tuple):
    def __new__(cls, *args):
        if len(args) == 1 and isinstance(args[0], str):
            channels = parse_hex_color(args[0])

        elif len(args) == 3:
            if any(isinstance(x, bool) for x in args):
                raise TypeError("color channels cannot be bool")

            channels = tuple(int(x) for x in args)

        else:
            raise TypeError("Color expects (r, g, b) or '#RRGGBB'")

        if not all(0 <= c <= 255 for c in channels):
            raise ValueError("channels must be in range 0..255")

        return super().__new__(cls, channels)

    r = property(lambda self: self[0])
    g = property(lambda self: self[1])
    b = property(lambda self: self[2])

    @property
    def hex(self):
        return f"#{self.r:02X}{self.g:02X}{self.b:02X}"

## Step 4 — Test equivalence, type, and hashing

In [107]:
a = Color(255, 128, 0)
b = Color("#ff8000")

print(a)
print(b)
print(a.hex)
print(a == b)
print(hash(a) == hash(b))

assert a == b
assert a.hex == "#FF8000"
assert isinstance(a, tuple)
assert type(a) is Color
assert hash(a) == hash(b)

(255, 128, 0)
(255, 128, 0)
#FF8000
True
True


# Problem 31 — Final mini-project: interned coordinate objects

Now combine:

- normalization
- immutable-looking public API
- weak-reference caching
- subclass-safe cache keys
- guarded initialization

We want repeated equivalent coordinates to reuse identity.

## Step 1 — Decide the identity key

We will convert both coordinates to floats and use:

```python
(cls, x, y)
```

as the cache key.

Including `cls` keeps subclasses independent.

In [108]:
import weakref

class Coordinate:
    _cache = weakref.WeakValueDictionary()

    def __new__(cls, x, y):
        x = float(x)
        y = float(y)
        key = (cls, x, y)

        existing = cls._cache.get(key)
        if existing is not None:
            return existing

        obj = super().__new__(cls)
        obj._x = x
        obj._y = y
        obj._initialized = False
        cls._cache[key] = obj
        return obj

    def __init__(self, x, y):
        if self._initialized:
            return

        self._initialized = True

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __repr__(self):
        return f"{type(self).__name__}({self.x}, {self.y})"

## Step 2 — Test identity reuse

In [109]:
a = Coordinate(1, 2)
b = Coordinate(1.0, 2.0)
c = Coordinate(2, 1)

print(a)
print("a is b:", a is b)
print("a is c:", a is c)

assert a is b
assert a is not c

Coordinate(1.0, 2.0)
a is b: True
a is c: False


## Step 3 — Test subclass separation

In [110]:
class ScreenCoordinate(Coordinate):
    pass

base = Coordinate(1, 2)
screen1 = ScreenCoordinate(1, 2)
screen2 = ScreenCoordinate(1.0, 2.0)

print(base is screen1)
print(screen1 is screen2)

assert base is not screen1
assert screen1 is screen2

False
True


# Review — Common `__new__` mistakes

Let us collect the most important failure modes.

## Mistake 1 — Forgetting to return

```python
def __new__(cls):
    super().__new__(cls)
```

Result: implicit `None`.

Fix:

```python
return super().__new__(cls)
```

## Mistake 2 — Trying to change immutable value in `__init__`

For immutable built-ins, the stored value must usually be created correctly in `__new__`.

## Mistake 3 — Hard-coding a base class

Fragile:

```python
return object.__new__(BaseClass)
```

More subclass-friendly:

```python
return super().__new__(cls)
```

## Mistake 4 — Reusing objects without considering `__init__`

If `__new__` returns an existing instance, `__init__` may run again.

Initialization must be safe and often idempotent.

## Mistake 5 — Cache key ignores the concrete class

A cache key like:

```python
normalized_value
```

may cause base classes and subclasses to share instances accidentally.

A safer key is often:

```python
(cls, normalized_value)
```

## Mistake 6 — Calling the class recursively inside `__new__`

```python
return cls(value)
```

normally re-enters `__new__` and causes recursion.

Delegate to the appropriate parent constructor instead.

# Final best-practice checklist

Before overriding `__new__`, ask:

### 1. Do I really need to control creation?

If normal attribute initialization is enough, use `__init__`.

### 2. Am I subclassing an immutable built-in?

If yes, `__new__` is often exactly the right place.

### 3. Does inheritance need to cooperate?

Prefer `super().__new__(cls, ...)` when the MRO should participate.

### 4. Am I returning an instance of the requested class?

If not, remember that the requested class's `__init__` will not normally run.

### 5. Am I caching/reusing objects?

Then think carefully about repeated `__init__` calls.

### 6. Does my cache work correctly for subclasses?

Consider including `cls` in the cache key.

### 7. Did I test both value and concrete type?

For `__new__`, both matter.

### 8. Does serialization matter?

Test pickle round trips for custom immutable constructor signatures.

### 9. Is a factory method clearer?

If construction chooses unrelated concrete types, a named factory is often easier to understand.

### 10. Is the implementation simpler than the problem?

`__new__` is powerful, but it should solve an actual object-creation requirement rather than merely add cleverness.

# Extra practice — solve these yourself

These deliberately do not include full implementations immediately after them.

## Exercise A — `EvenInt`

Create an `int` subclass that converts odd values to the next larger even integer.

Examples:

```python
EvenInt(4) == 4
EvenInt(5) == 6
```

---

## Exercise B — `NonEmptyTuple`

Create a `tuple` subclass that rejects empty construction.

---

## Exercise C — `NormalizedEmail`

Create a `str` subclass that:

- trims whitespace
- lowercases the domain part
- validates that exactly one `@` exists

Decide whether the local part should be lowercased or preserved, and document your choice.

---

## Exercise D — weakly interned `Tag`

Equivalent normalized tag strings should reuse identity while still allowing unused objects to be garbage-collected.

---

## Exercise E — subclass-safe singleton

Design a singleton base class where each subclass gets its *own* singleton instance.

---

## Exercise F — immutable `Percentage`

Store the normalized fraction internally:

```python
Percentage(25) -> 0.25
```

but display it as `"25.00%"`.

---

## Exercise G — immutable date key

Create a tuple subclass representing `(year, month, day)` with validation.

---

## Exercise H — constructor trace challenge

Build three inheritance levels, override `__new__` at each level, and predict the exact call order before running the code.

# Closing perspective

The most useful way to think about `__new__` is:

> `__new__` answers the question: **What object should this class call return?**

Sometimes the answer is:

- a newly allocated instance
- a transformed immutable built-in
- a previously cached instance
- a subclass-aware object
- or even an object of another type

Once that question is clear, the role of `__init__` also becomes clearer:

> `__init__` configures an already-created compatible instance.

That distinction is the foundation for reasoning about advanced Python object construction.